# 📱 Image Enhancer — Web App (phone-friendly)

Same Real-ESRGAN + GFPGAN engine, wrapped in a clean web page you can open on your phone.

**How to use:**
1. `Runtime → Change runtime type → T4 GPU → Save`
2. Run **Cell 1** (~2 min, once per session)
3. Run **Cell 2** — it prints a public link like `https://xxxx.gradio.live`
4. Open that link on your phone: upload from camera roll → Enhance → long-press the result to save

**Good to know:**
- The link only works while this Colab session is running. If it disconnects, just re-run Cell 2 for a fresh link.
- The first enhancement takes ~20–30 s (models download and load); after that it's faster.

In [ ]:
# Cell 1 — Install everything (~2 minutes, run once per session)
!pip install -q realesrgan gfpgan basicsr facexlib gradio

# Patch basicsr for newer torchvision (the functional_tensor module was renamed)
import glob
for f in glob.glob('/usr/local/lib/python3*/dist-packages/basicsr/data/degradations.py'):
    s = open(f).read().replace(
        'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
        'from torchvision.transforms.functional import rgb_to_grayscale')
    open(f, 'w').write(s)

print('✅ Installed — now run Cell 2')

In [ ]:
# Cell 2 — Launch the web app (open the gradio.live link it prints)
import cv2, torch
import gradio as gr
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
from gfpgan import GFPGANer

WEIGHTS = {
    'Photo': ('https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth', 23),
    'Anime': ('https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth', 6),
}

_upsamplers = {}
_face_enhancers = {}

def get_upsampler(model_name):
    if model_name not in _upsamplers:
        url, blocks = WEIGHTS[model_name]
        net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                      num_block=blocks, num_grow_ch=32, scale=4)
        # tile=400 keeps memory in check on the free T4 GPU
        _upsamplers[model_name] = RealESRGANer(
            scale=4, model_path=url, model=net,
            tile=400, tile_pad=10, pre_pad=0,
            half=torch.cuda.is_available())
    return _upsamplers[model_name]

def get_face_enhancer(model_name, scale):
    key = (model_name, scale)
    if key not in _face_enhancers:
        _face_enhancers[key] = GFPGANer(
            model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth',
            upscale=scale, arch='clean', channel_multiplier=2,
            bg_upsampler=get_upsampler(model_name))
    return _face_enhancers[key]

def enhance(image, model_name, scale_label, face_enhance):
    if image is None:
        raise gr.Error('Upload an image first.')
    scale = int(scale_label.replace('x', ''))
    img = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    if face_enhance:
        _, _, output = get_face_enhancer(model_name, scale).enhance(
            img, has_aligned=False, only_center_face=False, paste_back=True)
    else:
        output, _ = get_upsampler(model_name).enhance(img, outscale=scale)
    return cv2.cvtColor(output, cv2.COLOR_BGR2RGB)

app = gr.Interface(
    fn=enhance,
    inputs=[
        gr.Image(type='numpy', label='Upload image'),
        gr.Radio(['Photo', 'Anime'], value='Photo', label='Model'),
        gr.Radio(['2x', '4x'], value='4x', label='Upscale'),
        gr.Checkbox(value=True, label='Face enhancement (GFPGAN)'),
    ],
    outputs=gr.Image(type='numpy', label='Enhanced'),
    title='✨ Image Enhancer',
    description='Real-ESRGAN + GFPGAN on a free Colab GPU. Upload → Enhance → long-press the result to save. First run takes ~20–30 s while models load.',
)

app.launch(share=True)